RAG PIPELINE- Data Ingestion to Data Retrieval Pipeline

In [21]:
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
import faiss
from faiss import IndexFlatL2
import joblib
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
import re


In [22]:
def load_data(directory:str='D:/RAG-CHATBOT/pdf_directory'):
    all_documents=[]
    dir=Path(directory)

    all_files=list(dir.glob('**/*.pdf'))

    for file in all_files:
        print(f'Loading {file.name}')

        try:
            loader=PyMuPDFLoader(str(file))
            documents=loader.load()
            
            for doc in documents:
                doc.metadata['source']=file.name
                doc.metadata['type']='pdf'
                all_documents.append(doc)
        except Exception as e:
            print(f'Error as {e}')
    print(f' Total length of files {len(all_documents)}')
    return all_documents



In [23]:
def split_documents(all_documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n','\n',',',';','!','.']
    )

    split_docs=text_splitter.split_documents(all_documents)
    return split_docs

In [24]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

In [25]:
 #STEP 3: Load embedding model (just a plain function, no class)
# -------------------------------------------------------------------
from numpy import float32
from sentence_transformers import SentenceTransformer

def Convert_text(model_name:str='all-MiniLM-L6-v2'):
    model=SentenceTransformer(model_name)
    return model

In [26]:
def embeddings_generation(model,split_docs):
    text=[doc.page_content for doc in split_docs]
    embeddings=model.encode(text)
    embeddings=np.array(embeddings).astype(float32)

    return embeddings


In [27]:
# STEP 5: Build FAISS index
# -------------------------------------------------------------------
def Vector_Database(embeddings):
    dimension=embeddings.shape[1]
    index=faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return index

In [28]:
def _tokenize(text):
    return re.findall(r'\w+',text.lower())
    

In [29]:
def bm25_index(split_docs):
    tokenized_corpus=[_tokenize(doc.page_content) for doc in split_docs]
    return BM25Okapi(tokenized_corpus)


In [30]:
cross_encoder=CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieval(query,model,index,split_docs,k=3,fetch_k=15):
    query_embeddings=np.array(model.encode([query])).astype(float32)

    _,indices=index.search(query_embeddings,fetch_k)

    vector_indices=[  i for i in indices[0]]
    
    bm25_scores=bm25.get_scores(_tokenize(query))
    bm_indices=list(np.argsort(bm25_scores)[::-1] [:fetch_k])

    candidate_indices=list(dict.fromkeys(vector_indices+bm_indices))

    pairs=[[query,split_docs[i].page_content] for i in candidate_indices]
    rerank_scores=cross_encoder.predict(pairs)

    ranked=sorted(zip(candidate_indices,rerank_scores), key=lambda x:x[1],reverse=True)
    top_indices=[idx for idx,_ in ranked[:k]]

    results=[split_docs[i].page_content for i in top_indices]

    return results


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2550.07it/s]


In [31]:
from groq import Groq
import os
from dotenv import load_dotenv
groq_api_key=os.getenv('GROQ_API_KEY')

client = Groq(api_key=groq_api_key)  

def generate_answer(query, results):
    context = "\n\n".join(results)

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",   
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.3,
        max_tokens=1000
    )

    return response.choices[0].message.content

In [32]:
import joblib


all_documents = load_data()
split_docs = split_documents(all_documents)
model = Convert_text()
embeddings = embeddings_generation(model, split_docs)
index = Vector_Database(embeddings)
bm25=bm25_index(split_docs)
cross_encoder=CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

joblib.dump(split_docs, 'split_docs.pkl')
joblib.dump(index, 'index.pkl')
joblib.dump(bm25,'bm25.pkl')

print("Saved split_docs.pkl,index.pkl and bm25.pkl")

Loading about_gaurav-1.pdf
 Total length of files 3


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2928.57it/s]


Saved split_docs.pkl,index.pkl and bm25.pkl
